In [0]:
# ============================================================
# Bronze — Source 06: Stripe API
#
# Reads raw Stripe JSON files from S3 Raw landing zone
# and writes them as Delta tables to the Bronze layer.
#
# Source: s3://ecommerce-lakehouse-467091806172-raw-01/source=06_stripe/
# Target: bronze.stripe catalog in Unity Catalog
#
# Tables:
#   - bronze.stripe.charges
#
# Raw format: JSON (hourly partitioned, charges array per file)
# ============================================================

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")

RAW_BUCKET = "s3://ecommerce-lakehouse-467091806172-raw-01"
SOURCE = "06_stripe"

In [0]:
# Read all Stripe charges JSON files across all days and hours
# Each file contains a "charges" array — explode is needed in next step
from pyspark.sql.functions import explode, col

charges_raw = spark.read.format("json") \
    .option("multiLine", "false") \
    .load(f"{RAW_BUCKET}/source={SOURCE}/year=2026/month=04/day=*/hour=*/stripe_charges_*.json")

print(f"Raw file count (one per hour): {charges_raw.count()}")

In [0]:
# Explode charges array — each file has multiple charges inside
# Flatten nested structs: billing_details, metadata, dispute
# metadata.customer_id and metadata.order_id are the join keys

charges_flat = charges_raw \
    .select(explode(col("charges")).alias("charge")) \
    .select(
        col("charge.payment_intent_id"),
        col("charge.amount"),
        col("charge.amount_captured"),
        col("charge.amount_refunded"),
        col("charge.currency"),
        col("charge.status"),
        col("charge.captured"),
        col("charge.description"),
        col("charge.created"),
        col("charge.created_at"),
        col("charge.metadata.customer_id").alias("customer_id"),
        col("charge.metadata.order_id").alias("order_id"),
        col("charge.failure_code"),
        col("charge.failure_message"),
        col("charge.livemode"),
        col("charge.paid"),
        col("charge.billing_details.email").alias("billing_email"),
        col("charge.billing_details.name").alias("billing_name"),
        col("charge.billing_details.address.city").alias("billing_city"),
        col("charge.billing_details.address.country").alias("billing_country"),
        col("charge.billing_details.address.postal_code").alias("billing_postcode"),
        col("charge.dispute.id").alias("dispute_id"),
        col("charge.dispute.amount").alias("dispute_amount"),
        col("charge.dispute.reason").alias("dispute_reason"),
    )

print(f"Flattened charges count: {charges_flat.count()}")

# Create schema and write to Bronze Delta table
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.stripe")

charges_flat.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.stripe.charges")

print("✅ bronze.stripe.charges written")
spark.sql("SELECT COUNT(*) as row_count FROM bronze.stripe.charges").show()